# Preparación de CORe diario y mensual para todas las provincias (uso del docente)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/herramientas/reanalisis/preparar_core.ipynb)

1. De cada archivo de 6 horas de CORe descarga **solo** los bytes de los 8 campos necesarios (≈0,4 de 1,6 MB), recorta Ecuador y arma el día de 01:00 a 01:00 hora local.
2. Reparte el resultado en **un archivo diario y uno mensual por provincia**, que se publican en el release `core-v1`.

**Cuidado del servidor de CPC:** 4 hilos, pausa aleatoria entre solicitudes, reintentos con espera creciente (respeta 429/503) y reutilización del inventario `.idx`. Para 1981–2024 son ~64 000 archivos de 6 h; cuente con varias horas por década. Es reanudable por año.


| Variable | Columna | Unidad | ERA5 | CORe |
|---|---|---|---|---|
| Temperatura media, máxima, mínima a 2 m | `tmed`, `tmax`, `tmin` | °C | Estadísticos diarios de `2m_temperature` | Media de TMP 2 m; máx. de TMAX; mín. de TMIN |
| Precipitación | `pr` | mm/día | Suma diaria de `total_precipitation` | Suma de PRATE × 21 600 s |
| Humedad relativa | `hr` | % | De T y punto de rocío medios (Magnus) | De SPFH, T y presión de cada 6 h, promediada |
| Humedad específica | `q` | g/kg | De punto de rocío y presión medios | SPFH a 2 m, promediada |
| Presión en superficie | `ps` | hPa | Media de `surface_pressure` | Media de PRES sfc |
| Viento a 10 m | `u10`, `v10` | m/s | Medias de las componentes | Medias de UGRD y VGRD |


© 2026 Cristian Omar Paliz Acosta · ESPAM MFL · Todos los derechos reservados

In [ ]:
!pip -q install cfgrib eccodes regionmask

import os, sys, pathlib, subprocess
REPO_DIR = pathlib.Path(os.environ.get("CC_REPO", "/content/cambio-climatico-espam"))
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/crispaliz/cambio-climatico-espam", str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
from herramientas.reanalisis import era5, core, release
import xarray as xr
from google.colab import drive
drive.mount("/content/drive")

import datetime as dt, urllib.request
TRABAJO = pathlib.Path("/content/core_tmp")                         # GRIB temporales (se borran)
ANUAL = pathlib.Path("/content/drive/MyDrive/CORe_Ecuador/anual")     # un archivo por año, todo Ecuador
RELEASE = pathlib.Path("/content/drive/MyDrive/CORe_Ecuador/release") # un archivo por provincia
print("Recuadro de Ecuador [N, O, S, E]:", core.RECUADRO_ECUADOR)

## 1. Prueba con un día

In [ ]:
url = core.url_6h(dt.datetime(2000, 1, 1, 6))
sel = core.seleccion(core.leer_inventario(urllib.request.urlopen(url + ".idx", timeout=60).read().decode()))
print("Rangos de bytes:", core.rangos(sel))
datos, estado = core._get(url, core.rangos(sel)[0])
print("Respuesta del servidor:", estado, "(206 = descarga parcial correcta; 200 = ignora rangos y manda todo)")

d = core.dia(dt.date(2000, 1, 1), TRABAJO, borrar_grib=False)
print(d)
print({v: round(float(d[v].mean()), 2) for v in d.data_vars})
d["tmed"].isel(time=0).plot()

Revise: respuesta **206**; Tmax > Tmed > Tmin; precipitación ≥ 0 mm/día; HR entre 0 y 100 %; q de unos 15–20 g/kg en la Costa y menos en la Sierra; q entre ~5 y 20 g/kg en la Costa y menor en la Sierra; presión en hPa; viento de pocos m/s. Malla de ~1,9°: unas 6 latitudes por 13 longitudes en el recuadro.

## 2. Periodo completo

In [ ]:
ANIOS = range(1981, 2025)
anuales = core.preparar_diario(ANIOS, TRABAJO, ANUAL, hilos=4)

## 3. Un archivo diario y uno mensual por provincia

In [ ]:
archivos = release.por_provincia("core", anuales, RELEASE, mensual=core.mensual_desde_diario)
for f in archivos: print(f"{f.name:45s} {f.stat().st_size / 1e6:6.1f} MB")

## 4. Publicar

En GitHub: **Releases › Draft a new release**, etiqueta `core-v1`, y adjunte los archivos de `Mi unidad/CORe_Ecuador/release`. Cite a NOAA/NCEP CPC como fuente de CORe.